In [5]:
import sys
import subprocess
import urllib.request
from pathlib import Path
import importlib

# Only the dependencies needed by the remote client and our analysis.
subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "httpx>=0.28.1,<0.29", "tqdm", "pandas", "matplotlib"
])

# Load the official remote client without building local database extensions.
source_root = Path("/content/woc_remote_source")
package_dir = source_root / "woc"
package_dir.mkdir(parents=True, exist_ok=True)

base_url = "https://raw.githubusercontent.com/ssc-oscar/python-woc/master/woc/"

for filename in ["base.py", "remote.py"]:
    with urllib.request.urlopen(base_url + filename, timeout=30) as response:
        (package_dir / filename).write_bytes(response.read())

(package_dir / "__init__.py").write_text(
    "# Remote-only copy of the official python-woc client.\n"
)

# Ensure imports use this remote-only copy.
sys.path.insert(0, str(source_root))
for name in list(sys.modules):
    if name == "woc" or name.startswith("woc."):
        del sys.modules[name]

importlib.invalidate_caches()
from woc.remote import WocMapsRemote

print("\nREADY — World of Code remote client imported successfully.")


READY — World of Code remote client imported successfully.


In [6]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
from woc.remote import WocMapsRemote
import pandas as pd
import sqlite3
import json
import time

NETID = "gwright30"
OUT = Path("/content/drive/MyDrive") / f"COSC445_MP2_{NETID}"
OUT.mkdir(parents=True, exist_ok=True)

assignment_url = (
    "https://raw.githubusercontent.com/"
    "GreyWright/MiniProject2/master/net2prj.csv"
)

assignments = pd.read_csv(assignment_url)
assigned = assignments[
    assignments["netID"].str.strip().str.lower() == NETID
].copy()

assert len(assigned) == 10, (
    f"Expected 10 projects, found {len(assigned)}. Stop and check NetID."
)

assigned.to_csv(
    OUT / f"{NETID}_assigned_projects.csv",
    index=False
)
projects = assigned["WoC"].str.strip().tolist()

woc = WocMapsRemote(base_url="https://worldofcode.org/api/")

db = sqlite3.connect(str(OUT / "commit_cache.sqlite"))
db.execute("""
    CREATE TABLE IF NOT EXISTS commits (
        sha TEXT PRIMARY KEY,
        author TEXT NOT NULL,
        time INTEGER NOT NULL,
        message TEXT NOT NULL
    )
""")
db.commit()

print("Saving progress to:", OUT)
display(assigned)

Mounted at /content/drive
Saving progress to: /content/drive/MyDrive/COSC445_MP2_gwright30


,netID,WoC,GH
700,gwright30,juliamanifolds_manifolds.jl,https://github.com/JuliaManifolds/Manifolds.jl
701,gwright30,pydata_pandas-datareader,https://github.com/pydata/pandas-datareader
702,gwright30,shellphish_driller,https://github.com/shellphish/driller
703,gwright30,threeme3_qcx-ssb,https://github.com/threeme3/QCX-SSB
704,gwright30,humanbrainproject_fairgraph,https://github.com/humanbrainproject/fairgraph
705,gwright30,sancus-pma_sancus-core,https://github.com/sancus-pma/sancus-core
706,gwright30,vcflib_vcflib,https://github.com/vcflib/vcflib
707,gwright30,mobleylab_blues,https://github.com/MobleyLab/blues
708,gwright30,vinecopulib_rvinecopulib,https://github.com/vinecopulib/rvinecopulib
709,gwright30,catboost_catboost,https://github.com/catboost/catboost


In [8]:
def retry_request(function, *args):
    for attempt in range(4):
        try:
            return function(*args)
        except Exception as exc:
            if attempt == 3:
                raise
            delay = 5 * (attempt + 1)
            print(f"Retrying in {delay}s: {exc}", flush=True)
            time.sleep(delay)


# Retrieve and cache each project's complete commit-ID list.
def get_commit_ids(project):
    return sorted(set(woc.iter_values("p2c", project)))


project_commits = {}

for project in projects:
    cache_file = OUT / f"{project}_sha_list.json"

    if cache_file.exists():
        hashes = json.loads(cache_file.read_text())
    else:
        hashes = retry_request(get_commit_ids, project)

        if not hashes:
            raise RuntimeError(f"No commits returned for {project}.")

        temporary = cache_file.with_suffix(".tmp")
        temporary.write_text(json.dumps(hashes))
        temporary.replace(cache_file)
        time.sleep(1)

    project_commits[project] = hashes
    print(f"{project}: {len(hashes):,} commits", flush=True)


# Skip commits already saved by an earlier run.
all_hashes = sorted({
    sha
    for hashes in project_commits.values()
    for sha in hashes
})

saved = {row[0] for row in db.execute("SELECT sha FROM commits")}
pending = [sha for sha in all_hashes if sha not in saved]

print(
    f"\nUnique commits: {len(all_hashes):,}\n"
    f"Already saved: {len(set(all_hashes) & saved):,}\n"
    f"Still needed: {len(pending):,}\n",
    flush=True,
)


def fetch_batch(batch):
    results, errors = woc.get_values_many("commit.tch", batch)
    missing = [sha for sha in batch if not results.get(sha)]

    if missing:
        # Try the separate WoC commit-content endpoint.
        time.sleep(1)
        fallback, fallback_errors = woc.show_content_many(
            "commit", missing
        )

        for sha in missing:
            if fallback.get(sha):
                results[sha] = [fallback[sha]]
            else:
                # Record unavailable data explicitly.
                with (OUT / "missing_commits.jsonl").open("a") as log:
                    log.write(json.dumps({
                        "sha": sha,
                        "map_error": str(errors.get(sha, "")),
                        "content_error": str(
                            fallback_errors.get(sha, "")
                        ),
                    }) + "\n")

                print(
                    f"Logged unavailable commit: {sha}",
                    flush=True,
                )

    return results


started = time.monotonic()
BATCH_SIZE = 10
saved_this_run = 0
unavailable_this_run = 0

for start in range(0, len(pending), BATCH_SIZE):
    batch = pending[start:start + BATCH_SIZE]
    results = retry_request(fetch_batch, batch)

    rows = []

    for sha in batch:
        if not results.get(sha):
            unavailable_this_run += 1
            continue

        commit = results[sha][0]
        rows.append((
            sha,
            commit[2][0],
            int(commit[2][1]),
            commit[4],
        ))

    # Save each successful batch to the Drive-backed database.
    with db:
        db.executemany(
            "INSERT OR REPLACE INTO commits VALUES (?, ?, ?, ?)",
            rows,
        )

    saved_this_run += len(rows)
    done = min(start + BATCH_SIZE, len(pending))
    elapsed = time.monotonic() - started

    # Include the pause following this batch in the estimate.
    remaining_minutes = (
        (len(pending) - done) * (elapsed + 1) / done / 60
    )

    if start == 0 or done % 100 == 0 or done == len(pending):
        print(
            f"Processed {done:,}/{len(pending):,} | "
            f"Saved {saved_this_run:,} new commits | "
            f"Unavailable {unavailable_this_run:,} | "
            f"Estimated remaining: {remaining_minutes:.1f} min",
            flush=True,
        )

    time.sleep(1)


# Associate each commit with its correct project.
membership = pd.DataFrame(
    [
        (project, sha)
        for project, hashes in project_commits.items()
        for sha in hashes
    ],
    columns=["project_wocid", "commit_sha1"],
)

details = pd.read_sql_query(
    """
    SELECT sha AS commit_sha1,
           author,
           time,
           message AS "commit message"
    FROM commits
    """,
    db,
)

summary = membership.merge(
    details,
    on="commit_sha1",
    how="left",
    validate="many_to_one",
)

assert membership["project_wocid"].nunique() == 10
assert not summary.duplicated(
    ["project_wocid", "commit_sha1"]
).any()


# Export missing IDs separately so they can be investigated.
missing_rows = summary[summary["time"].isna()].copy()
missing_file = OUT / f"{NETID}_missing_commits.csv"

missing_rows[["project_wocid", "commit_sha1"]].to_csv(
    missing_file,
    sep=";",
    index=False,
)

summary = summary[summary["time"].notna()].copy()
summary["time"] = summary["time"].astype("int64")
summary = summary.sort_values(
    ["project_wocid", "time", "commit_sha1"]
)

output_file = OUT / f"{NETID}_project_summary.csv"
summary.to_csv(output_file, sep=";", index=False)


# Show expected versus retrieved counts for all ten projects.
counts = pd.DataFrame({
    "Expected": membership.groupby("project_wocid").size(),
    "Retrieved": summary.groupby("project_wocid").size(),
}).fillna(0).astype(int)

counts["Missing"] = counts["Expected"] - counts["Retrieved"]

print(f"\nCOLLECTION PASS FINISHED: {len(summary):,} rows saved.")
print("Summary:", output_file)

if len(missing_rows):
    print(
        f"FOLLOW-UP NEEDED: {len(missing_rows):,} project-commit "
        f"rows unavailable. IDs saved to {missing_file}"
    )
else:
    print("All listed commits were retrieved.")

display(counts)

juliamanifolds_manifolds.jl: 8,262 commits
pydata_pandas-datareader: 1,758 commits
shellphish_driller: 568 commits
threeme3_qcx-ssb: 461 commits
humanbrainproject_fairgraph: 1,793 commits
sancus-pma_sancus-core: 459 commits
vcflib_vcflib: 1,764 commits
mobleylab_blues: 1,659 commits
vinecopulib_rvinecopulib: 1,230 commits
catboost_catboost: 51,191 commits

Unique commits: 69,145
Already saved: 1,070
Still needed: 68,075

Logged unavailable commit: 03fb1ee0f14ae00a125b56052dc6a84b58844ffc
Processed 10/68,075 | Saved 9 new commits | Unavailable 1 | Estimated remaining: 269.9 min
Processed 100/68,075 | Saved 99 new commits | Unavailable 1 | Estimated remaining: 139.1 min
Logged unavailable commit: 049e3e749d9ce0cd09a60c3356cffc7d8a33d7c7
Processed 200/68,075 | Saved 198 new commits | Unavailable 2 | Estimated remaining: 137.4 min
Processed 300/68,075 | Saved 298 new commits | Unavailable 2 | Estimated remaining: 132.7 min
Processed 400/68,075 | Saved 398 new commits | Unavailable 2 | Esti

,Expected,Retrieved,Missing
project_wocid,,,
catboost_catboost,51191,51141,50
humanbrainproject_fairgraph,1793,1793,0
juliamanifolds_manifolds.jl,8262,8262,0
mobleylab_blues,1659,1657,2
pydata_pandas-datareader,1758,1750,8
sancus-pma_sancus-core,459,459,0
shellphish_driller,568,568,0
threeme3_qcx-ssb,461,459,2
vcflib_vcflib,1764,1759,5


In [2]:
from google.colab import drive, files
from pathlib import Path
import zipfile

drive.mount("/content/drive")

folder = Path("/content/drive/MyDrive/COSC445_MP2_gwright30")
required = [
    folder / "gwright30_project_summary.csv",
    folder / "gwright30_missing_commits.csv",
    folder / "gwright30_assigned_projects.csv",
]

missing = [path.name for path in required if not path.is_file()]

if missing:
    print("Expected folder:", folder)
    print("Folder exists:", folder.exists())
    print("Missing files:", missing)

    if folder.exists():
        print("Files present:", [p.name for p in folder.iterdir()])

    raise RuntimeError("Required files not found in the expected folder.")

zip_path = Path("/content/gwright30_collected_data_checked.zip")

with zipfile.ZipFile(
    zip_path, "w", compression=zipfile.ZIP_DEFLATED
) as archive:
    for path in required:
        archive.write(path, arcname=path.name)

    log = folder / "missing_commits.jsonl"
    if log.is_file():
        archive.write(log, arcname=log.name)

with zipfile.ZipFile(zip_path) as archive:
    for item in archive.infolist():
        print(f"{item.filename}: {item.file_size:,} bytes")

print(f"\nZIP size: {zip_path.stat().st_size:,} bytes")
files.download(str(zip_path))

Mounted at /content/drive
gwright30_project_summary.csv: 14,910,713 bytes
gwright30_missing_commits.csv: 4,075 bytes
gwright30_assigned_projects.csv: 729 bytes
missing_commits.jsonl: 18,518 bytes

ZIP size: 4,904,839 bytes


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## GitHub observations — October 8, 2026

I checked all ten project repositories on GitHub. Most of the star and fork counts were visible directly,
but CatBoost and pandas-datareader had rounded figures on the website, so I used the GitHub API
to get the exact counts for those two. GitHub API snapshots are also saved with the data
so the recorded counts and branch information can be checked. Commit dates below are the
latest default-branch committer dates in UTC, as recorded on October 8, 2026.

| Project | Stars | Forks | Latest GitHub commit |
|---|---:|---:|---|
| juliamanifolds_manifolds.jl | 436 | 72 | 2026-09-30 |
| pydata_pandas-datareader | 3272 | 693 | 2026-07-21 |
| shellphish_driller | 984 | 163 | 2025-03-24 |
| threeme3_qcx-ssb | 861 | 263 | 2025-08-26 |
| humanbrainproject_fairgraph | 14 | 10 | 2026-10-06 |
| sancus-pma_sancus-core | 24 | 15 | 2025-05-19 |
| vcflib_vcflib | 686 | 223 | 2026-10-02 |
| mobleylab_blues | 34 | 16 | 2021-01-22 |
| vinecopulib_rvinecopulib | 40 | 11 | 2026-09-18 |
| catboost_catboost | 9133 | 1345 | 2026-10-07 |

QCX-SSB redirects to threeme3/usdx, and sancus-pma/sancus-core
redirects to sancus-tee/sancus-core. The assigned WoC IDs are retained.

## World of Code collection summary

The collection saved 69,077 of 69,145 listed commits across my ten
assigned projects. The remaining 68 commits were unavailable from both
the commit lookup and the fallback endpoint; their IDs were saved
separately in gwright30_missing_commits.csv.

The table below describes the original WoC collection, before the
separate GitHub recovery pass. Authors are counted as distinct author
strings, so aliases may count separately. Dates are author timestamps
converted to UTC.

| Project | Retrieved commits | Authors | Earliest commit (UTC) | Latest commit (UTC) |
|---|---:|---:|---|---|
| juliamanifolds_manifolds.jl | 8262 | 88 | 2019-06-05 18:27:27 | 2026-04-24 14:59:59 |
| pydata_pandas-datareader | 1750 | 203 | 2014-12-06 19:46:57 | 2025-09-24 18:22:20 |
| shellphish_driller | 568 | 35 | 2015-06-13 06:01:25 | 2025-03-24 19:44:27 |
| threeme3_qcx-ssb | 459 | 20 | 2019-01-28 17:44:15 | 2025-10-31 21:10:28 |
| humanbrainproject_fairgraph | 1793 | 14 | 2018-03-22 15:47:20 | 2025-11-03 21:58:18 |
| sancus-pma_sancus-core | 459 | 18 | 2011-03-04 20:17:50 | 2025-05-19 08:19:58 |
| vcflib_vcflib | 1759 | 118 | 2010-09-22 23:26:17 | 2025-11-04 09:36:40 |
| mobleylab_blues | 1657 | 45 | 2016-06-28 00:20:31 | 2025-10-15 20:00:56 |
| vinecopulib_rvinecopulib | 1229 | 11 | 2017-03-30 18:54:11 | 2025-10-20 12:43:10 |
| catboost_catboost | 51141 | 1550 | 2017-07-18 05:33:21 | 2025-11-08 18:22:21 |

## Collection method and limitations

Installing python-woc failed in the Colab Python 3.13 environment.
The setup cell loads the official remote-client Python modules directly,
avoiding compilation of the local database extensions.

Commit IDs were retrieved using p2c. Commit details were collected in
batches of ten, with pauses between requests and a Drive-backed SQLite
checkpoint so interrupted collection could resume.

The supplied data extend beyond the January 2025 cutoff stated in the
assignment README. Coverage varies by project. WoC history can also
differ from the current GitHub default-branch history.